## Import / setup


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np

# 將共整合目錄新增到路徑以匯入 get_data 模組
sys.path.insert(0, str(Path.cwd()))

from get_data import (
    create_bybit_session,
    get_all_bybit_usdt_spot_symbols,
    get_all_bybit_perp_symbols,
    fetch_sector_map_with_report,
    download_binance_sector_data,
    explore_downloaded_data,
    preview_market_data,
)

from coint import (
    load_basis_matrices,
    load_funding_matrix,
    load_sector_map,
    rolling_sector_cointegration_scan,
    walk_forward_basis_backtest,
    build_trading_log,
    summarize_backtest,
)

DATABASE_PATH = Path("data")
CONFIG_PATH = Path("config/api_config.json")  # 選填：用於 API 認證
N_JOBS = -1  # -1 表示使用所有 CPU cores
SHOW_PROGRESS = True
ROLLING_PAIRS_CACHE = DATABASE_PATH / "cache" / "rolling_basis_pairs.parquet"

START_DATE = datetime(2020, 1, 1)
END_DATE = datetime.now()

print(f"數據收集範圍: {START_DATE.date()} 至 {END_DATE.date()}")
print(f"資料庫路徑: {DATABASE_PATH}")
print("數據格式: Parquet")

try:
    bybit_session = create_bybit_session()
    print("Bybit 會話創建成功")
except ImportError as e:
    print(e)
    bybit_session = None

try:
    spot_symbols = get_all_bybit_usdt_spot_symbols()
    perp_symbols = get_all_bybit_perp_symbols()
    print(f"已獲取 {len(spot_symbols)} 個 Bybit 現貨 USDT 符號，範例: {spot_symbols[:5]}")
    print(f"已獲取 {len(perp_symbols)} 個 Bybit 永續 USDT 符號，範例: {perp_symbols[:5]}")
except Exception as e:
    print(f"獲取 Bybit 符號出錯: {e}")
    spot_symbols = []
    perp_symbols = []

數據收集範圍: 2020-01-01 至 2026-05-27
資料庫路徑: data
數據格式: Parquet
Bybit 會話創建成功
已獲取 447 個 Bybit 現貨 USDT 符號，範例: ['BTCUSDT', 'ETHUSDT', 'XRPUSDT', 'DOTUSDT', 'XLMUSDT']
已獲取 570 個 Bybit 永續 USDT 符號，範例: ['0GUSDT', '1000000BABYDOGEUSDT', '1000000CHEEMSUSDT', '1000000MOGUSDT', '10000NEXUSDT']


# Data

## Fetch Sector Classification


In [ ]:
try:
    fetch_sector_map_with_report(
        base_path=DATABASE_PATH,
        categories_to_process=("spot", "linear"),
        sleep_seconds=0.5,
    )
except ImportError as e:
    print(e)
except Exception as e:
    print(f"獲取行業映射出錯: {e}")

## Download Binance Sector Data


In [ ]:
skip_sectors = ["USD Stablecoin", "Fiat-backed Stablecoin"]

try:
    binance_universe = download_binance_sector_data(
        base_path=DATABASE_PATH,
        start_date=START_DATE,
        end_date=END_DATE,
        skip_sectors=skip_sectors,
        interval="1h",
        sleep_seconds=0.5,
        include_funding_rate=False,
    )
except Exception as e:
    print(f"下載過程中出錯: {e}")

## Data Exploration and Validation


In [25]:
data_summaries = explore_downloaded_data(DATABASE_PATH)

Data exploration and validation

Found 228 Parquet files in data\spot:

  1. 0GUSDT.parquet
     Shape: 5905 rows x 10 columns
     Columns: ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', '#trade', 'taker_buy_vol', 'week']
     Dtypes: {'open_time': 'datetime64[ns]', 'open': 'float64', 'high': 'float64', 'low': 'float64', 'close': 'float64', 'volume': 'float64', 'close_time': 'datetime64[ns]', '#trade': 'int64', 'taker_buy_vol': 'float64', 'week': 'int32'}
     Date range: 2025-09-22 10:00:00 to 2026-05-26 10:00:00

  2. 1INCHUSDT.parquet
     Shape: 47464 rows x 10 columns
     Columns: ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', '#trade', 'taker_buy_vol', 'week']
     Dtypes: {'open_time': 'datetime64[ns]', 'open': 'float64', 'high': 'float64', 'low': 'float64', 'close': 'float64', 'volume': 'float64', 'close_time': 'datetime64[ns]', '#trade': 'int64', 'taker_buy_vol': 'float64', 'week': 'int32'}
     Date range: 2020-12-25 05:00:00 to

## Preview Parquet Data


In [24]:
preview_market_data(DATABASE_PATH)

Loaded 0GUSDT from spot

Full data shape: (5905, 10)

First 5 rows:
            open_time   open   high    low  close       volume  \
0 2025-09-22 10:00:00  1.000  5.186  1.000  3.494  10116834.93   
1 2025-09-22 11:00:00  3.497  7.115  3.461  5.608  18383684.46   
2 2025-09-22 12:00:00  5.601  7.260  5.050  5.523  10927671.89   
3 2025-09-22 13:00:00  5.514  5.765  4.600  5.337   8212414.02   
4 2025-09-22 14:00:00  5.336  5.584  4.755  4.893   4596974.75   

               close_time  #trade  taker_buy_vol  week  
0 2025-09-22 10:59:59.999  104427     4817770.08     0  
1 2025-09-22 11:59:59.999  235935     9432368.73     0  
2 2025-09-22 12:59:59.999  169646     5601447.52     0  
3 2025-09-22 13:59:59.999  138080     4183596.86     0  
4 2025-09-22 14:59:59.999  115465     2213203.85     0  

Basic statistics:
              open        close        volume
count  5905.000000  5905.000000  5.905000e+03
mean      1.059152     1.059138  2.612517e+05
std       0.812981     0.813116  5.8

# Analysis

In [ ]:
# Rolling sector-level futures-spot basis cointegration scan
spot_dir = DATABASE_PATH / "spot"
futures_dir = DATABASE_PATH / "futures"
funding_dir = DATABASE_PATH / "funding_rate"
metadata_path = DATABASE_PATH / "metadata" / "top10_market_sector_map.json"

basis, spot_prices, futures_prices = load_basis_matrices(
    spot_dir=spot_dir,
    futures_dir=futures_dir,
    price_column="close",
    basis_method="log",
    min_obs=500,
)
funding_rates = load_funding_matrix(
    funding_dir,
    symbols=basis.columns,
    min_obs=1,
)
sector_map = load_sector_map(
    metadata_path,
    skip_sectors=["USD Stablecoin", "Fiat-backed Stablecoin"],
    available_symbols=basis.columns,
)

print(f"basis 矩陣: {basis.shape[0]} 筆時間資料 x {basis.shape[1]} 個幣種")
print(f"funding rate 矩陣: {funding_rates.shape[0]} 筆時間資料 x {funding_rates.shape[1]} 個幣種")
print(f"可分析板塊數: {len(sector_map)}")

rolling_basis_pairs = rolling_sector_cointegration_scan(
    prices=basis,
    sector_map=sector_map,
    formation_window="60D",
    trading_window="7D",
    step="7D",
    max_pvalue=0.05,
    max_spread_adf_pvalue=0.05,
    min_obs=1000,
    top_n_per_sector=3,
    n_jobs=N_JOBS,
    cache_path=ROLLING_PAIRS_CACHE,
    use_cache=True,
    show_progress=SHOW_PROGRESS,
)

rolling_basis_backtests, rolling_basis_summaries = walk_forward_basis_backtest(
    basis=basis,
    spot_prices=spot_prices,
    futures_prices=futures_prices,
    rolling_pairs=rolling_basis_pairs,
    funding_rates=funding_rates,
    max_pairs_per_window=10,
    entry_z=2.0,
    exit_z=0.5,
    fee_rate=0.0004,
)

basis_trading_log = build_trading_log(rolling_basis_backtests)

print(f"rolling basis pair 數量: {len(rolling_basis_pairs)}")
print(f"回測 pair-window 數量: {len(rolling_basis_summaries)}")
print(f"交易筆數: {len(basis_trading_log)}")

if rolling_basis_summaries.empty:
    print("沒有 rolling window 產生可回測的 basis 配對，請放寬 p-value、縮短 min_obs，或確認期現資料期間足夠。")
else:
    display(rolling_basis_pairs.head(20))
    display(rolling_basis_summaries.sort_values(["window_id", "sharpe"], ascending=[True, False]).head(20))
    basis_trading_log.head(50)

basis 矩陣: 56075 筆時間資料 x 192 個幣種
funding rate 矩陣: 14970 筆時間資料 x 191 個幣種
可分析板塊數: 8


rolling windows:   0%|          | 0/326 [00:00<?, ?it/s]

window 1: sectors:   0%|          | 0/8 [00:00<?, ?it/s]

window 1: Smart Contract Platform pairs (6786):   0%|          | 0/21 [00:00<?, ?it/s]

window 1: Layer 1 (L1) pairs (2850):   0%|          | 0/21 [00:00<?, ?it/s]

...

window 68: Stablecoins pairs (6): 0it [00:00, ?it/s]

window 68: Made in USA pairs (2628):   0%|          | 0/528 [00:00<?, ?it/s]

window 68: Exchange-based Tokens pairs (496):   0%|          | 0/45 [00:00<?, ?it/s]

window 69: sectors:   0%|          | 0/8 [00:00<?, ?it/s]

window 69: Smart Contract Platform pairs (6786):   0%|          | 0/861 [00:00<?, ?it/s]